In [1]:
import os
import pandas as pd
from itertools import product 
from dotenv import load_dotenv
from mosqlient import upload_prediction

# Load environment variables from .env file
load_dotenv()

# Access the environment variables
api_key = os.getenv('api_key')

In [2]:

geocodes = [2931350, 2933307, 2302503, 3119401, 3549805,
           3541406, 1200401, 1200203, 1716109, 4113700, 4103701, 4104808,
            5201405, 5102637, 5215231]

disease = 'dengue'


In [3]:
if disease == 'dengue': 
    disease_code = 'A90'

In [5]:

for geo, test_year in product(geocodes, [2023, 2024, 2025, 2026]): 
    
    df = pd.read_csv(f'predictions/preds_{disease}_{geo}_{test_year}.csv')

    if test_year == 2023: 
        val_test =1
    elif test_year == 2024:
        val_test =2 
    elif test_year == 2025:
        val_test = 3
    elif test_year == 2026:
        val_test = 3

    # correção do df

    df['date'] = pd.to_datetime(df['date'])

    # ordenar
    df = df.sort_values('date')

    # usar como índice
    df = df.set_index('date')

    # criar índice semanal (domingos)
    full_index = pd.date_range(df.index.min(), df.index.max(), freq='W-SUN')

    # inserir semanas faltantes
    df = df.reindex(full_index)

    # preencher com a semana anterior
    df = df.fillna(df.shift(1))

    # voltar coluna date
    df = df.reset_index().rename(columns={'index': 'date'})


    pred = upload_prediction(
    api_key=api_key,
    disease="A90",
    repository="eduardocorrearaujo/3rd_imdc_emap_lstm_muni",
    commit="94763d415c0ba57dce9a8418093ff4a2b6a7a6b4",
    description = f'Prediction for optional challenge 1 - geocode:{geo}, and validation test {val_test}',
    case_definition="probable",
    published=True,
    adm_level=2,
    adm_0="BRA",
    adm_1=int(str(geo)[:2]),
    adm_2 = geo, 
    prediction=df 
)
    